# 05 — Train YOLO26n (face + license-plate, 1280×1280 tiles)

Trains **YOLO26n** from the official pretrained `yolo26n.pt` on the 2-class
(`face`, `license-plate`) tiled dataset.

**Why a dedicated nano / 2-class run:** this model is meant for fast, on-upload
anonymisation where only faces and license-plates must be found. Restricting to two
classes (instead of the full 10-class export) removes label noise from irrelevant
objects and lets the small backbone spend all its capacity on the two target classes.

**Configuration (as specified for this run):**
| Item | Value |
|---|---|
| Model | **YOLO26n** (init `yolo26n.pt`, official pretrained) |
| imgsz | **1280** (= tile size, no runtime resize, lossless PNG tiles) |
| Classes | **2**: `face`, `license-plate` (`dataset_face_lp/dataset.yaml`) |
| Epochs | **400** |
| Warmup epochs | **20** |
| Early stopping | **patience = 60** (stop after 60 epochs with no val improvement) |
| Batch | from `runs/optimal_batch.json` (measured by `04_batch_finder.ipynb`) |
| Optimizer | SGD, lr0=0.01, cosine LR, AMP |
| Output | `runs/faces/faces_yolo26n_v1/` |

The training cell is started as a background `nbconvert` process so it runs to
completion unattended; progress is tracked via `runs/faces/<run>/results.csv`.

In [ ]:
# --- pi-heif compatibility fix MUST run before importing ultralytics ---
import sys
sys.path.insert(0, '/home/jovyan/shared/s0598584/scripts')
import piheif_fix  # noqa: F401  (auto-applies on import)

import os, json, glob, multiprocessing
from pathlib import Path
import torch
from ultralytics import YOLO

ROOT = Path('/home/jovyan/shared/s0598584')
YAML_PATH = ROOT/'dataset_face_lp'/'dataset.yaml'   # nc=2 face + license-plate
INIT_WEIGHTS = str(ROOT/'yolo26n.pt')               # official pretrained nano

IMGSZ    = 1280
EPOCHS   = 400
WARMUP   = 20
PATIENCE = 60

# Batch size: measured by the real batch-finder (04). Fail loudly if absent — we must
# NOT silently fall back to a guessed value (that was the old bug).
_ob = ROOT/'runs'/'optimal_batch.json'
assert _ob.exists(), 'runs/optimal_batch.json missing — run 04_batch_finder.ipynb first!'
_obj = json.loads(_ob.read_text())
assert _obj.get('model') == 'yolo26n' and _obj.get('imgsz') == IMGSZ, \
    f'optimal_batch.json is for a different config: {_obj.get("model")} @ {_obj.get("imgsz")}'
BATCH = int(_obj['optimal_batch'])

device  = 0
# workers=4: images are cached in RAM (see train cell), so there is no NFS latency to
# hide and a few workers suffice. Keeping the count low also keeps the startup fork +
# AMP-check spike well under the ~48 GiB RAM cgroup (more workers OOM-killed the kernel).
workers = min(4, multiprocessing.cpu_count())

assert YAML_PATH.exists(), f'dataset.yaml missing: {YAML_PATH}'
assert Path(INIT_WEIGHTS).exists(), f'init weights missing: {INIT_WEIGHTS}'
def _count(split): return len(glob.glob(str(ROOT/'dataset_face_lp'/'images'/split/'*.png')))
n_train, n_val, n_test = _count('train'), _count('val'), _count('test')
assert n_train > 0 and n_val > 0 and n_test > 0, 'a split is empty!'

print('='*64)
print('init weights :', INIT_WEIGHTS)
print('dataset      :', YAML_PATH)
print('imgsz        :', IMGSZ, '(no resize)')
print('batch        :', BATCH, f"(measured; peak {_obj.get('peak_vram_gb')} / {_obj.get('total_vram_gb')} GB)")
print('epochs       :', EPOCHS, '| warmup:', WARMUP, '| patience:', PATIENCE)
print('device       :', device, '| workers:', workers)
if torch.cuda.is_available():
    print('GPU          :', torch.cuda.get_device_name(0))
print('train/val/test images:', n_train, '/', n_val, '/', n_test)
print('='*64)

model = YOLO(INIT_WEIGHTS)
print('Model loaded — ready to train.')

In [ ]:
# --- Training ---
EXPERIMENT = 'faces_yolo26n_v1'
OUTPUT_DIR = ROOT/'runs'/'faces'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

results = model.train(
    data=str(YAML_PATH),
    imgsz=IMGSZ,
    epochs=EPOCHS,
    patience=PATIENCE,
    device=device,
    batch=BATCH,
    workers=workers,
    save=True,
    save_period=50,
    project=str(OUTPUT_DIR),
    name=EXPERIMENT,
    exist_ok=True,
    # The 1280-px tiles live on a slow network volume. We cache the decoded images in RAM
    # once (~18 GB for this 2-class dataset, well under the box's RAM budget), so every
    # epoch is served from memory and training is GPU-bound instead of NFS-I/O bound.
    cache='ram',
    # --- optimisation ---
    optimizer='SGD',
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=WARMUP,
    cos_lr=True,
    amp=True,
    # --- augmentation ---
    # Photometric augmentation was already done offline (notebook 02). At train time we
    # keep light geometric + colour jitter. mosaic/mixup/copy_paste are DISABLED: mosaic
    # downscales objects, which hurts recall on the small faces/plates we target (and it
    # would also multiply the cached-image lookups). flipud=0 — faces/plates are never
    # upside-down in street imagery.
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10.0, translate=0.1, scale=0.3, shear=2.0, perspective=0.0002,
    flipud=0.0, fliplr=0.5,
    mosaic=0.0, close_mosaic=0, mixup=0.0, copy_paste=0.0,
    # --- eval/plots ---
    conf=0.001, iou=0.5, plots=True, verbose=True,
)
print('Training finished!')